# Présentation du jeu de données

Le jeu de données utilisé dans ce tutoriel est celui de l’inventaire forestier national allemand, disponible [ici](https://www.openagrar.de/receive/openagrar_mods_00094435).

Certaines parties de ce notebook sont également issues du notebook original fourni par les créateurs des données [ici](https://www.openagrar.de/servlets/MCRFileNodeServlet/openagrar_derivate_00058265/intro_to_dataset.ipynb)

Pour plus d’informations sur ce jeu de données, veuillez consulter l’article [A Sentinel-2 machine learning dataset for tree species classification in Germany](https://essd.copernicus.org/articles/17/351/2025/)

Ce jeu de données comprend les valeurs de réflectance au sommet de l’atmosphère pour la majorité des arbres de l’inventaire forestier national allemand de 2012.

Les attributs de ce jeu de données sont :

![attributes](https://essd.copernicus.org/articles/17/351/2025/essd-17-351-2025-t01-thumb.png)

In [2]:
import math
import sqlite3
import datetime
import numpy as np
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
import geopandas as gpd

## Chargement et formattage des données

Le jeu de données se présente sous la forme d’un fichier SQLite. Un fichier SQLite est un fichier unique, autonome et multiplateforme, qui contient une *base de données SQLite*. Nous commençons par établir une connexion à SQLite et nous demandons à SQLite de renvoyer les chaînes de caractères sous forme de tableaux d’octets. Les données sont stockées dans la table "data" de la base sqlite.

In [ ]:
dataset_path = "S2GNFI_V1" # change this path to point to the database file after you download and unzip it from the source given above
conn = sqlite3.connect(dataset_path)
conn.text_factory = bytes  # this makes sqlite return strings as bytes that we can parse via numpy

Pour charger l’ensemble du jeu de données, il faudrait environ 30 Go de RAM !
Mais grâce à SQLite, nous pouvons limiter le chargement à un sous-ensemble afin de simplement en inspecter une partie.
Dans le cadre de ce tutoriel, nous nous restreignons à trois classes d’arbres : l’épinette de Norvège (identifié par le nombre 10), le pin sylvestre (identifié par le nombre 20) et le hêtre (identifié par le nombre 100). Nous appliquons également quelques filtres supplémentaires pour ne conserver que les données les plus récemment vérifiées, et nous gardons uniquement les variables qui semblent pertinentes pour la classification des arbres.

Le code suivant sélectionne une partie de la base de données SQLite qui satisfait les filtres définis dans la variable "query", puis les charge dans une structure de données tabulaire à deux dimensions appelée DataFrame.

In [ ]:
query = """
SELECT
    species,
    boa,
    qai,
    dbh_mm,
    height_dm,
    crown_area_m2,
    time,
    tree_id,
    X_WGS84,
    Y_WGS84
FROM data
WHERE species IN (10, 100, 20)
  AND present_2022 = 1
  AND is_corrected = 1
  AND is_pure = 1
"""
df = pd.read_sql_query(query, conn)
print("initial shape of the dataset:", df.shape)
df.head(10) # to see the first 10 items in our dataframe

initial shape of the dataset: (9878978, 10)


,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,b'\xda\x00`\x01\xeb\x00\x07\x02\xee\x05\xfc\x0...,8192,254,201,12.342376,1438732800,94895,9.268243,48.126625
1,10,b'\x01\x01~\x01\xf3\x00\x14\x02\xd0\x05\xef\x0...,8192,254,201,12.342376,1439251200,94895,9.268243,48.126625
2,10,"b'\x84\x00\xfe\x00\x83\x00\x8f\x01,\x05\x8e\x0...",8192,254,201,12.342376,1440460800,94895,9.268243,48.126625
3,10,b'r\x00\xe1\x00\x7f\x00d\x01\xbe\x04\xf2\x05x\...,8192,254,201,12.342376,1441065600,94895,9.268243,48.126625
4,10,b'\xd6\x00\xd7\x00\xcd\x00S\x01\x93\x03\x05\x0...,10240,254,201,12.342376,1449360000,94895,9.268243,48.126625
5,10,b'(\x00\x9c\x00g\x00B\x01T\x04{\x05X\x05\xe9\x...,10240,254,201,12.342376,1453680000,94895,9.268243,48.126625
6,10,b'\x95\x00-\x01\xc3\x00\xea\x01\xf3\x04\xe4\x0...,8192,254,201,12.342376,1462147200,94895,9.268243,48.126625
7,10,b'3\x01\xc0\x01\x11\x01?\x02=\x06=\x08\x9a\x08...,8192,254,201,12.342376,1466726400,94895,9.268243,48.126625
8,10,b'\x19\x01\x95\x01\x12\x01\x1c\x02P\x06/\x08\r...,8192,254,201,12.342376,1470441600,94895,9.268243,48.126625
9,10,b'n\x00\xe2\x00m\x00u\x01S\x05\xca\x06\xb8\x06...,8192,254,201,12.342376,1470960000,94895,9.268243,48.126625


Les réflectances corrigées des effets atmosphériques (BOA) semblent encore étranges ! Cela s’explique par le fait qu’elles sont affichées sous forme brute (bytes). Nous pouvons facilement convertir les 20 octets en entiers de base 10 signés sur 16 bits.

Vous pouvez trouver un aperçu des bandes Sentinel-2 [ici](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/bands/).

<table><tbody><tr><th>Band</th><th>Resolution</th><th>Central Wavelength</th><th>Description</th></tr><tr><td>B2</td><td>10 m</td><td>490 nm</td><td>Blue</td></tr><tr><td>B3</td><td>10 m</td><td>560 nm</td><td>Green</td></tr><tr><td>B4</td><td>10 m</td><td>665 nm</td><td>Red</td></tr><tr><td>B5</td><td>20 m</td><td>705 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B6</td><td>20 m</td><td>740 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B7</td><td>20 m</td><td>783 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B8</td><td>10 m</td><td>842 nm</td><td>Near Infrared (VNIR)</td></tr><tr><td>B8a</td><td>20 m</td><td>865 nm</td><td>Near Infrared (VNIR)</td></tr><tr><td>B11</td><td>20 m</td><td>1610 nm</td><td>Short Wave Infrared (SWIR)</td></tr><tr><td>B12</td><td>20 m</td><td>2190 nm</td><td>Short Wave Infrared (SWIR)</td></tr></tbody></table>

In [ ]:
df.boa = [np.frombuffer(x, dtype=np.int16) for x in df.boa]
df.head(10)

,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",8192,254,201,12.342376,1438732800,94895,9.268243,48.126625
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",8192,254,201,12.342376,1439251200,94895,9.268243,48.126625
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",8192,254,201,12.342376,1440460800,94895,9.268243,48.126625
3,10,"[114, 225, 127, 356, 1214, 1522, 1656, 1682, 6...",8192,254,201,12.342376,1441065600,94895,9.268243,48.126625
4,10,"[214, 215, 205, 339, 915, 1285, 1156, 1285, 36...",10240,254,201,12.342376,1449360000,94895,9.268243,48.126625
5,10,"[40, 156, 103, 322, 1108, 1403, 1368, 1513, 38...",10240,254,201,12.342376,1453680000,94895,9.268243,48.126625
6,10,"[149, 301, 195, 490, 1267, 1508, 1493, 1647, 7...",8192,254,201,12.342376,1462147200,94895,9.268243,48.126625
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",8192,254,201,12.342376,1466726400,94895,9.268243,48.126625
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",8192,254,201,12.342376,1470441600,94895,9.268243,48.126625
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",8192,254,201,12.342376,1470960000,94895,9.268243,48.126625


C’est mieux ! Mais qu’en est-il de la date ? Convertissons-la du format unix à un format plus compréhensible pour un humain.

In [ ]:
df.time = pd.to_datetime(df.time, unit="s", utc=True)
df.head(10)

,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",8192,254,201,12.342376,2015-08-05 00:00:00+00:00,94895,9.268243,48.126625
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",8192,254,201,12.342376,2015-08-11 00:00:00+00:00,94895,9.268243,48.126625
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",8192,254,201,12.342376,2015-08-25 00:00:00+00:00,94895,9.268243,48.126625
3,10,"[114, 225, 127, 356, 1214, 1522, 1656, 1682, 6...",8192,254,201,12.342376,2015-09-01 00:00:00+00:00,94895,9.268243,48.126625
4,10,"[214, 215, 205, 339, 915, 1285, 1156, 1285, 36...",10240,254,201,12.342376,2015-12-06 00:00:00+00:00,94895,9.268243,48.126625
5,10,"[40, 156, 103, 322, 1108, 1403, 1368, 1513, 38...",10240,254,201,12.342376,2016-01-25 00:00:00+00:00,94895,9.268243,48.126625
6,10,"[149, 301, 195, 490, 1267, 1508, 1493, 1647, 7...",8192,254,201,12.342376,2016-05-02 00:00:00+00:00,94895,9.268243,48.126625
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",8192,254,201,12.342376,2016-06-24 00:00:00+00:00,94895,9.268243,48.126625
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",8192,254,201,12.342376,2016-08-06 00:00:00+00:00,94895,9.268243,48.126625
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",8192,254,201,12.342376,2016-08-12 00:00:00+00:00,94895,9.268243,48.126625


## Filtrage de la qualité

Commençons par ne conserver que les échantillons d’arbres provenant des mois d’été. Puisque ces données proviennent d’Allemagne, nous savons que toutes les latitudes sont positives et que la saison d'été correspond à la période de juin à août, mais gardons notre filtre général et supposons que nous pourrions également avoir des données provenant de l’hémisphère sud.

In [ ]:
df = df[
    ((df.Y_WGS84 >= 0) & df.time.dt.month.between(6, 8)) |
    ((df.Y_WGS84 < 0) & df.time.dt.month.isin([12, 1, 2]))
]
# Nous allons maintenant ne retenir que le jour de l'année et ne plus prendre en compte l'heure exacte.
df["doy"] = df.time.dt.dayofyear
df = df.drop(columns=["time"])
print("shape of the dataset after seasonal filtering:", df.shape)
print(df.head(10))

shape of the dataset after seasonal filtering: (3146686, 10)
    species                                                boa   qai  dbh_mm  \
0        10  [218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...  8192     254   
1        10  [257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...  8192     254   
2        10  [132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...  8192     254   
7        10  [307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...  8192     254   
8        10  [281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...  8192     254   
9        10  [110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...  8192     254   
10       10  [76, 198, 102, 347, 1293, 1585, 1735, 1796, 62...  8192     254   
12       10  [173, 260, 177, 391, 1289, 1575, 1784, 1854, 6...  8192     254   
27       10  [160, 324, 167, 530, 1548, 1732, 1879, 1860, 7...  8192     254   
28       10  [212, 393, 218, 550, 1777, 2033, 2187, 2191, 8...  8192     254   

    height_dm  crown_area_m2  tree_id   X_WGS84    Y_WGS84

Notez qu’il existe une colonne QAI, signifiant « informations sur l’assurance qualité ». Cette colonne contient des données générées à la suite du traitement [FORCE](https://force-eo.readthedocs.io/) des images satellites. Chaque condition particulière de l’image est encodée à l’aide d’un bit différent dans un entier non signé sur 16 bits, comme expliqué [ici](https://force-eo.readthedocs.io/en/latest/howto/qai).

Pour plus de clarté, voici les différents masques de bits en code Python :

In [ ]:
VALID        = 0b000000000000000
NODATA       = 0b000000000000001
CLOUD_BUFFER = 0b000000000000010
CLOUD_OPAQUE = 0b000000000000100
CLOUD_CIRRUS = 0b000000000000110
CLOUD_SHADOW = 0b000000000001000
SNOW         = 0b000000000010000
WATER        = 0b000000000100000
AOD_INT      = 0b000000001000000
AOD_HIGH     = 0b000000010000000
AOD_FILL     = 0b000000011000000
SUBZERO      = 0b000000100000000
SATURATION   = 0b000001000000000
SUN_LOW      = 0b000010000000000
ILLUMIN_LOW  = 0b000100000000000
ILLUMIN_POOR = 0b001000000000000
ILLUMIN_NONE = 0b001100000000000
SLOPED       = 0b010000000000000
WVP_NONE     = 0b100000000000000

Examinons à présent la valeur QAI du premier enregistrement.

In [ ]:
df.iloc[0]["qai"]

8192

... et convertissons cette valeur en représentation binaire :

In [ ]:
bin(df.iloc[0]["qai"])

'0b10000000000000'

Nous pouvons à présent repérer la position des chiffres 1 à partir de la droite : le seul chiffre 1 se trouve à la 14ᵉ position. Selon le tableau ci-dessus, cela indique que la zone est en pente. Vérifions cela :

In [ ]:
SLOPED

8192

Exact!

Effectuons une vérification rapide: est-ce que FORCE a identifié certains pixels de notre jeu de données comme étant de l’eau ?

In [ ]:
my_filter = WATER

Si l’on souhaite déterminer si certaines valeurs QAI correspondent à notre filtre, il suffit de vérifier la présence d’un 1 aux mêmes positions que celles exprimées dans le notre filtre. Pour ce faire, on applique l'opérateur « ET logique » (« & »), qui retrourne un bit à 1 si ce bit est activé. Ainsi, toute correspondance produira automatiquement un nombre supérieur à zéro.

In [ ]:
mask = (df["qai"] & my_filter) > 0
mask

0          False
1          False
2          False
7          False
8          False
           ...  
9878971    False
9878972    False
9878973    False
9878974    False
9878975    False
Name: qai, Length: 3146686, dtype: bool

In [ ]:
np.sum(mask)

168

-> On trouve 168 cas dans ce sous-ensemble de données où FORCE a identifié de l’eau. Cette détection n’a évidemment pas de sens puisque les arbres mesurés se trouvent sur la terre ferme, mais ce résultat peut survenir en cas de pluie.

Nous allons maintenant exclure toutes les observations associées aux nuages, à la neige ou à d’autres conditions météorologiques défavorables, en comparant chaque valeur QAI à un filtre à l’aide de l’opérateur "ET logique". Si les valeurs QAI et le filtre n’ont aucun bit en commun, la valeur obtenue est zéro : ce sont précisément les enregistrements « propres » qui nous intéressent.

On peut également utiliser l’opérateur « OU logique » (« | ») pour regrouper plusieurs bits : « a | b » donne un bit à 1 si ce bit est activé dans l’une des deux valeurs.

In [ ]:
# On utilise « OU logique » (« | ») pour fusionner les filtres, puis « ET logique » (« & ») pour les comparer aux valeurs QAI, si le résultat est zéro, l’enregistrement est valide.
qa_filter = NODATA | CLOUD_BUFFER | CLOUD_CIRRUS | CLOUD_OPAQUE | CLOUD_SHADOW | SNOW | WATER | AOD_HIGH | AOD_FILL | AOD_INT | ILLUMIN_POOR | SUN_LOW | SATURATION
df = df[(df["qai"] & qa_filter) == 0]
print("shape of the dataset after quality filtering:", df.shape)

shape of the dataset after quality filtering: (2653641, 10)


Éliminons la colonne relative à l’assurance qualité puisqu’elle n’est plus utile pour la classification.

In [ ]:
df = df.drop(columns=["qai"])

Sentinel-2 peut produire des valeurs de réflectance négatives au-dessus des arbres, en particulier dans les produits de niveau 2A (correspondant à la réflectance au sol). Pourtant, cela n’est pas physiquement possible. Ces valeurs négatives apparaissent à cause d’artéfacts liés à la correction atmosphérique sur des cibles très sombres, comme les ombres profondes dans des forêts denses de conifères, ou sur les plans d’eau, là où l’algorithme de correction surestime la diffusion atmosphérique.
Il est donc également préférable de détecter et de supprimer ces valeurs de notre jeu de données.

In [ ]:
df = df[df["boa"].apply(lambda x: min(x) >= 0)]
print("shape of the dataset after reflectance filtering:", df.shape)

shape of the dataset after reflectance filtering: (2644231, 9)


## NDVI

Pour approfondir l’exploration des données, nous pouvons examiner l’Indice de Végétation par différence Normalisée (NDVI). Nous définissons le NDVI à partir des bandes rouge et proche infrarouge. Nous choisissons d'utiliser la bande proche infrarouge à 842 nm, puisque cette dernière offre une résolution de 10 m, tout comme la bande rouge.

In [18]:
def ndvi(boa):
    boa32 = boa.astype(np.float32)
    return (boa32[6] - boa32[2]) / (boa32[6] + boa32[2] + np.float32(1e-8))

In [19]:
df["ndvi"] = df["boa"].apply(ndvi)
df.head(10)

,species,boa,dbh_mm,height_dm,crown_area_m2,tree_id,X_WGS84,Y_WGS84,doy,ndvi
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",254,201,12.342376,94895,9.268243,48.126625,217,0.783809
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",254,201,12.342376,94895,9.268243,48.126625,223,0.778386
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",254,201,12.342376,94895,9.268243,48.126625,237,0.863114
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",254,201,12.342376,94895,9.268243,48.126625,176,0.779394
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",254,201,12.342376,94895,9.268243,48.126625,219,0.765310
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",254,201,12.342376,94895,9.268243,48.126625,225,0.880809
10,10,"[76, 198, 102, 347, 1293, 1585, 1735, 1796, 62...",254,201,12.342376,94895,9.268243,48.126625,236,0.888949
12,10,"[173, 260, 177, 391, 1289, 1575, 1784, 1854, 6...",254,201,12.342376,94895,9.268243,48.126625,244,0.819480
27,10,"[160, 324, 167, 530, 1548, 1732, 1879, 1860, 7...",254,201,12.342376,94895,9.268243,48.126625,153,0.836755
28,10,"[212, 393, 218, 550, 1777, 2033, 2187, 2191, 8...",254,201,12.342376,94895,9.268243,48.126625,170,0.818711


Puisque nous avons déjà supprimé les valeurs aberrantes de notre jeu de données, les valeurs de NDVI devraient être comprises entre -1 et 1. Faisons néanmoins une dernière vérification et éliminons tous les échantillons dont le NDVI serait en hors de cet intervalle ou dont la valeur serait NaN.


In [20]:
df = df[df["ndvi"].between(-1, 1) & df["ndvi"].notna()]
print("shape of the dataset after ndvi filtering:", df.shape)

shape of the dataset after ndvi filtering: (2644231, 10)


## Séparation des données entre entraînement et test, puis sauvegarde du jeu de données

Séparons maintenant nos données en ensembles d’entraînement, de validation et de test pour la future tâche de classification.

*Pourquoi ne pas faire une division aléatoire ?*

Une division aléatoire risque de provoquer :

- Un chevauchement d’arbres (« tree leakage ») : le même arbre peut se retrouver dans l’entraînement et le test via plusieurs échantillons, ce qui gonfle artificiellement les métriques.
- Un chevauchement spatial (« spatial leakage ») : l’ensemble d’entraînement peut contenir des arbres très proches de ceux du test (autocorrélation spatiale), ce qui fausse aussi les métriques ; des stratégies de partitionnement spatial « par blocs » permettent de réduire ce biais.

Nous regroupons donc les échantillons par identifiant d’arbre et par proximité géographique, puis assignons ces groupes aux ensembles entraînement/test en fonction d’une proportion choisie (ex : 70 %/30 %).

In [21]:
def split_by_tree_geo(df):
    # tree-level table
    tree_tbl = (
        df.groupby("tree_id", as_index=False)
        .agg(lon=("X_WGS84", "mean"), lat=("Y_WGS84", "mean"), y=("species", "first"))
    )

    # Germany is primarily in the UTM zone 32N (EPSG code 25832) for its mapping and surveying purposes.
    gdf = gpd.GeoDataFrame(
        tree_tbl,
        geometry=gpd.points_from_xy(tree_tbl["lon"], tree_tbl["lat"]),
        crs="EPSG:4326"  # WGS84 (degrees)
    )
    
    # Germany-friendly projected CRS (meters)
    gdf = gdf.to_crs("EPSG:25832")  # ETRS89 / UTM zone 32N
    
    tree_tbl["x_m"] = gdf.geometry.x
    tree_tbl["y_m"] = gdf.geometry.y

    BLOCK_SIZE_M = 20000.0  # 20 km blocks (something I decided, but depending on our area, we could change it to be more meaninful)
    tree_tbl["bx"] = np.floor(tree_tbl["x_m"] / BLOCK_SIZE_M).astype(int)
    tree_tbl["by"] = np.floor(tree_tbl["y_m"] / BLOCK_SIZE_M).astype(int)
    tree_tbl["block_id"] = (tree_tbl["bx"].astype(str) + "_" + tree_tbl["by"].astype(str))

    # split blocks (groups = block_id)
    block_gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr_tree_idx, te_tree_idx = next(block_gss.split(tree_tbl, tree_tbl["y"], groups=tree_tbl["block_id"]))

    tree_tr = tree_tbl.iloc[tr_tree_idx].copy()
    tree_te = tree_tbl.iloc[te_tree_idx].copy()

    # split validation blocks inside train blocks
    block_gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr2_idx, va_idx = next(block_gss2.split(tree_tr, tree_tr["y"], groups=tree_tr["block_id"]))
    tree_va = tree_tr.iloc[va_idx].copy()
    tree_tr = tree_tr.iloc[tr2_idx].copy()

    train_tree_ids = set(tree_tr["tree_id"])
    val_tree_ids = set(tree_va["tree_id"])
    test_tree_ids = set(tree_te["tree_id"])

    df["split"] = "unused"

    df.loc[df.tree_id.isin(train_tree_ids), "split"] = "train"
    df.loc[df.tree_id.isin(val_tree_ids), "split"] = "val"
    df.loc[df.tree_id.isin(test_tree_ids), "split"] = "test"

    df_train = df[df.split == "train"]
    df_val   = df[df.split == "val"]
    df_test  = df[df.split == "test"]

    print("train, validtaion, test size")
    print(df_train.shape, df_val.shape, df_test.shape)

    print("train, validation and test unique tree_id counts: ")
    print(len(set(tree_tr.tree_id)), len(set(tree_va.tree_id)), len(set(tree_te.tree_id)))

    print("Tree overlap train/test:", len(set(df_train.tree_id) & set(df_test.tree_id)))
    print("Tree overlap train/val:", len(set(df_train.tree_id) & set(df_val.tree_id)))

    print("train, validation and test unique block_id counts: ")
    print(len(set(tree_tr.block_id)), len(set(tree_va.block_id)), len(set(tree_te.block_id)))
    print("Geographic overlap train/test:", len(set(tree_tr.block_id) & set(tree_te.block_id)))
    print("Geographic overlap train/val:", len(set(tree_tr.block_id) & set(tree_va.block_id)))


In [22]:
split_by_tree_geo(df)
df.head()

train, validtaion, test size
(1574978, 11) (343413, 11) (725840, 11)
train, validation and test unique tree_id counts: 
30306 7070 13766
Tree overlap train/test: 0
Tree overlap train/val: 0
train, validation and test unique block_id counts: 
417 105 225
Geographic overlap train/test: 0
Geographic overlap train/val: 0


,species,boa,dbh_mm,height_dm,crown_area_m2,tree_id,X_WGS84,Y_WGS84,doy,ndvi,split
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",254,201,12.342376,94895,9.268243,48.126625,217,0.783809,test
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",254,201,12.342376,94895,9.268243,48.126625,223,0.778386,test
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",254,201,12.342376,94895,9.268243,48.126625,237,0.863114,test
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",254,201,12.342376,94895,9.268243,48.126625,176,0.779394,test
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",254,201,12.342376,94895,9.268243,48.126625,219,0.765310,test


Nous sauvegardons ensuite ce DataFrame nettoyé dans un fichier Parquet afin de pouvoir le réutiliser plus tard. Le format Parquet est un format binaire ouvert orienté colonnes, développé par Apache. Ce format permet un stockage et une consultation des données très efficaces et particulièrement adapté à l’analyse de grandes quantités de données (« big data »). Contrairement aux formats traditionnels basés sur les lignes comme CSV ou JSON, qui stockent les données ligne par ligne, Parquet regroupe les données par colonne, ce qui apporte des avantages importants en termes de performance et d’espace de stockage (par exemple, la version CSV de ce jeu de données occuperait 350 Mo sur le disque, tandis que la version Parquet ne nécessiterait que 50 Mo).

In [23]:
save_path = "trees_dataset.parquet"
df.to_parquet(
    save_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
    row_group_size=500_000
)

En travaillant avec le format Parquet, on peut utiliser un chargement différé (lazy loading) et parcourir les données en flux (streaming) pour éviter de saturer la mémoire.

In [24]:
lf = pl.scan_parquet(save_path)

boa_labels = ["Blue", "Green", "Red", "B05", "B06", "B07", "NIR", "B8a", "SWIR1", "SWIR2"]

boa_cols = [pl.col("boa").list.get(i).cast(pl.Int32).alias(boa_labels[i]) for i in range(10)]
lf = lf.with_columns(boa_cols).drop("boa")

lf = lf.with_columns([
    pl.col("species").cast(pl.Int32),
    pl.col("tree_id").cast(pl.Int32),
    pl.col("dbh_mm").cast(pl.Float32),
    pl.col("height_dm").cast(pl.Float32),
    pl.col("crown_area_m2").cast(pl.Float32),
    pl.col("X_WGS84").cast(pl.Float32),
    pl.col("Y_WGS84").cast(pl.Float32),
    pl.col("doy").cast(pl.Int32),
    pl.col("ndvi").cast(pl.Float32),
    pl.col("split").cast(pl.String)
])

#this helps with later grouping by tree_id (we will see in next demo)
lf = (
    lf
    .sort("tree_id")
    .select(pl.all())  # forces column materialization
)

lf.sink_parquet(
    save_path,
    compression="zstd"
)

# Peek schema
lf.collect_schema()


Schema([('species', Int32),
        ('dbh_mm', Float32),
        ('height_dm', Float32),
        ('crown_area_m2', Float32),
        ('tree_id', Int32),
        ('X_WGS84', Float32),
        ('Y_WGS84', Float32),
        ('doy', Int32),
        ('ndvi', Float32),
        ('split', String),
        ('Blue', Int32),
        ('Green', Int32),
        ('Red', Int32),
        ('B05', Int32),
        ('B06', Int32),
        ('B07', Int32),
        ('NIR', Int32),
        ('B8a', Int32),
        ('SWIR1', Int32),
        ('SWIR2', Int32)])